In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [21]:
import os
import requests
import pandas as pd
import numpy as np
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta
load_dotenv()  # 환경변수 불러오기

True

# API 정보 불러오기

In [47]:
# 하이퍼 파라미터
# 111 : '쌀', 211 : '배추', 245 : '양파', 214 : '상추', 411 : '사과'
p_item_code='411'   
START_YEAR = 2001
END_YEAR = 2025


# KAMIS 인증 정보
CERT_KEY = os.getenv('kamis_key')
CERT_ID = os.getenv('kamis_id')

#아이템 코드
item_map = {'111' : '쌀',
           '211' : '배추',
           '245' : '양파',
           '214' : '상추',
           '411' : '사과'} 

# 대분류 지정
if p_item_code == '111':
    p_item_category_code = '100'
elif p_item_code in ('211', '245', '214'):
    p_item_category_code = '200'
elif p_item_code == '411':
    p_item_category_code = '400'
else:
    print('대분류에 없는 코드입니다')

url = 'https://www.kamis.or.kr/service/price/xml.do'

## 도매 가격 정보 조회

In [43]:
# 실행 코드
data_list = []
year = START_YEAR
for i in range(END_YEAR-START_YEAR+1):    
    p_startday = f'{year}-01-01'
    p_endday = f'{year}-12-31'
    if year == datetime.today().strftime('%Y'): # 올해라면, 전일자 까지만 뽑기
        p_endday= (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')

    params = {
        'action': 'periodWholesaleProductList',  # 신) 일별 도매 가격 자료
        'p_cert_key': CERT_KEY,
        'p_cert_id': CERT_ID,
        'p_returntype': 'json',
        'p_itemcategorycode': p_item_category_code,    # 대분류 코드 예: 200=과일류
        'p_itemcode': p_item_code,             # 품목코드 예: 211=사과
        'p_kindcode': '',              # 품종코드 예: 05=홍로
        # 'p_county_code': '1101',          # 지역코드 예: 서울=1101
        #'p_convert_kg_yn': 'N',
        'p_startday': p_startday,
        'p_endday': p_endday
    }

    # ✅ API 호출
    response = requests.get(url, params=params)
#     print("👉 호출 URL:", response.url)
#     print("👉 응답 상태 코드:", response.status_code)
#     print("👉 응답 내용 요약:", response.text[:500])
    # time.sleep(0.5)

    # ✅ 결과 확인 및 데이터 추가
    if response.status_code == 200:
        json_data = response.json()
        data = json_data.get('data')
        data_list.extend(data.get('item'))
    else:
        print(f"API 호출 실패 - 코드 : {response.status_code}, 작업중 : {item_map[p_item_code]} - {p_startday}~{p_endday}" )
        break

    # 다음 사이클로 이동
    year += 1
    print(f'작업완료 - {item_map[p_item_code]}, 기간 : {p_startday}~{p_endday}')
    time.sleep(0.5)

df = pd.DataFrame(data_list)
values = {'itemname' : item_map[p_item_code], 'kindname' : '-', 'marketname' : '-'}
df.fillna(value= values, inplace=True)
df.to_csv(f'data/kamis_api_도매_{item_map[p_item_code]}.csv', encoding='cp949')
print('=========저장완료==========')

작업완료 - 양파, 기간 : 2001-01-01~2001-12-31
작업완료 - 양파, 기간 : 2002-01-01~2002-12-31
작업완료 - 양파, 기간 : 2003-01-01~2003-12-31
작업완료 - 양파, 기간 : 2004-01-01~2004-12-31
작업완료 - 양파, 기간 : 2005-01-01~2005-12-31
작업완료 - 양파, 기간 : 2006-01-01~2006-12-31
작업완료 - 양파, 기간 : 2007-01-01~2007-12-31
작업완료 - 양파, 기간 : 2008-01-01~2008-12-31
작업완료 - 양파, 기간 : 2009-01-01~2009-12-31
작업완료 - 양파, 기간 : 2010-01-01~2010-12-31
작업완료 - 양파, 기간 : 2011-01-01~2011-12-31
작업완료 - 양파, 기간 : 2012-01-01~2012-12-31
작업완료 - 양파, 기간 : 2013-01-01~2013-12-31
작업완료 - 양파, 기간 : 2014-01-01~2014-12-31
작업완료 - 양파, 기간 : 2015-01-01~2015-12-31
작업완료 - 양파, 기간 : 2016-01-01~2016-12-31
작업완료 - 양파, 기간 : 2017-01-01~2017-12-31
작업완료 - 양파, 기간 : 2018-01-01~2018-12-31
작업완료 - 양파, 기간 : 2019-01-01~2019-12-31
작업완료 - 양파, 기간 : 2020-01-01~2020-12-31
작업완료 - 양파, 기간 : 2021-01-01~2021-12-31
작업완료 - 양파, 기간 : 2022-01-01~2022-12-31
작업완료 - 양파, 기간 : 2023-01-01~2023-12-31
작업완료 - 양파, 기간 : 2024-01-01~2024-12-31
작업완료 - 양파, 기간 : 2025-01-01~2025-12-31
=========저장완료==========


## 소매 가격 정보 조회

In [48]:
# 실행

data_list = []
year = START_YEAR
for i in range(END_YEAR-START_YEAR+1):    
    p_startday = f'{year}-01-01'
    p_endday = f'{year}-12-31'

    params = {
        'action': 'periodRetailProductList',  # 신) 일별 도매 가격 자료
        'p_cert_key': CERT_KEY,
        'p_cert_id': CERT_ID,
        'p_returntype': 'json',
        'p_itemcategorycode': p_item_category_code,    # 대분류 코드 예: 200=과일류
        'p_itemcode': p_item_code,             # 품목코드 예: 211=사과
        'p_kindcode': '',              # 품종코드 예: 05=홍로
        #'p_convert_kg_yn': 'N',
        'p_startday': p_startday,
        'p_endday': p_endday
    }

    # ✅ API 호출
    response = requests.get(url, params=params)
    # print("👉 호출 URL:", response.url)
    # print("👉 응답 상태 코드:", response.status_code)
    # print("👉 응답 내용 요약:", response.text[:500])
    # time.sleep(0.5)

    # ✅ 결과 확인 및 데이터 추가
    if response.status_code == 200:
        json_data = response.json()
        data = json_data.get('data')
        data_list.extend(data.get('item'))
    else:
        print(f"API 호출 실패 - 코드 : {response.status_code}, 작업중 : {item_map[p_item_code]} - {p_startday}~{p_endday}" )
        break

    # 다음 사이클로 이동
    year += 1
    print(f'작업완료 - {item_map[p_item_code]}, 기간 : {p_startday}~{p_endday}')
    time.sleep(0.5)
df = pd.DataFrame(data_list)
values = {'itemname' : item_map[p_item_code], 'kindname' : '-', 'marketname' : '-'}
df.fillna(value= values, inplace=True)
df.to_csv(f'data/kamis_api_소매_{item_map[p_item_code]}.csv', encoding='cp949')
print('=========저장완료==========')

작업완료 - 사과, 기간 : 2001-01-01~2001-12-31
작업완료 - 사과, 기간 : 2002-01-01~2002-12-31
작업완료 - 사과, 기간 : 2003-01-01~2003-12-31
작업완료 - 사과, 기간 : 2004-01-01~2004-12-31
작업완료 - 사과, 기간 : 2005-01-01~2005-12-31
작업완료 - 사과, 기간 : 2006-01-01~2006-12-31
작업완료 - 사과, 기간 : 2007-01-01~2007-12-31
작업완료 - 사과, 기간 : 2008-01-01~2008-12-31
작업완료 - 사과, 기간 : 2009-01-01~2009-12-31
작업완료 - 사과, 기간 : 2010-01-01~2010-12-31
작업완료 - 사과, 기간 : 2011-01-01~2011-12-31
작업완료 - 사과, 기간 : 2012-01-01~2012-12-31
작업완료 - 사과, 기간 : 2013-01-01~2013-12-31
작업완료 - 사과, 기간 : 2014-01-01~2014-12-31
작업완료 - 사과, 기간 : 2015-01-01~2015-12-31
작업완료 - 사과, 기간 : 2016-01-01~2016-12-31
작업완료 - 사과, 기간 : 2017-01-01~2017-12-31
작업완료 - 사과, 기간 : 2018-01-01~2018-12-31
작업완료 - 사과, 기간 : 2019-01-01~2019-12-31
작업완료 - 사과, 기간 : 2020-01-01~2020-12-31
작업완료 - 사과, 기간 : 2021-01-01~2021-12-31
작업완료 - 사과, 기간 : 2022-01-01~2022-12-31
작업완료 - 사과, 기간 : 2023-01-01~2023-12-31
작업완료 - 사과, 기간 : 2024-01-01~2024-12-31
작업완료 - 사과, 기간 : 2025-01-01~2025-12-31
=========저장완료==========
